# Retrieval isolado

O primeiro passo da fase de desenvolvimento é validar o `search_client` sozinho sem LLM e sem orquestração para confirmar que a busca híbrida (vetorial + semântica) traz, dentro do `top_k`, o conteúdo necessário para uma resposta correta. Se o chunk certo não está no `top_k`, nenhuma engenharia de prompt resolve depois, pois o modelo de geração nunca vê a informação.

**Corpus:** os 5 documentos em [`data/manual/`](../data/manual/): runbook de triagem, catálogo de sistemas, manual de escalonamento, postmortems de incidentes anteriores e guia de casos-limite.

Os exemplos abaixo foram escolhidos a dedo para ilustrar, cada um, uma capacidade diferente do retrieval. Não são só "perguntas fáceis", mas cenários que aparecem de verdade numa triagem de incidentes.

In [9]:
from maf_demo.search_client import retrieve

In [10]:
def show_chunks(chunks, preview_chars=420):
    for chunk in chunks:
        print(f"[{chunk.reranker_score:.2f}] {chunk.source} — {chunk.section_title}")
        print(chunk.content[:preview_chars].replace("\n", " "))
        print()

> **Como ler o score:** `reranker_score` vem do *semantic ranker* do Azure AI Search, numa escala de 0 a 4. Como referência de leitura para os exemplos abaixo: acima de ~2.5 é sinal forte e específico; entre ~1.5 e 2.5 ainda é relevante mas mais genérico; abaixo disso o chunk normalmente é ruído de cauda. O que mais importa, porém, não é o valor absoluto, é a **folga entre o 1º colocado e o resto**: uma folga grande indica que o retrieval está confiante sobre qual chunk responde a pergunta.

## Exemplo 1 — Sintoma óbvio → sistema e severidade óbvios

Caso mais simples possível: o sintoma usa quase as mesmas palavras do manual. Serve de linha de base: se o retrieval não acerta isso, nada mais importa.

**Esperado:** o topo deve vir do **Catálogo de Sistemas Críticos**, seção **Tier 0**, no trecho do **Auth Gateway** que lista literalmente "não consigo entrar" como sintoma típico de falha, o que por definição é SEV-1.

In [17]:
show_chunks(retrieve("login caiu, ninguém consegue entrar no sistema"))

[2.20] 02-catalogo-sistemas-criticos.md — Tier 0 — Caminho crítico de compra
## Tier 0 — Caminho crítico de compra  Sistemas cuja falha total impede diretamente uma venda de acontecer. Qualquer indisponibilidade completa aqui é, por definição, SEV-1.  **Auth Gateway** — responsável por login, criação de conta, recuperação de senha e emissão de tokens de sessão. Fica na frente de praticamente todo o resto: se ele cai, o cliente não consegue nem ver o próprio carrinho salvo, mesmo que todo o re

[1.97] 04-postmortems-incidentes-anteriores.md — Postmortems de Incidentes Anteriores — NimbusFlow
# Postmortems de Incidentes Anteriores — NimbusFlow  **Propósito:** registro histórico de incidentes já classificados e resolvidos, usado como referência para triagem de casos parecidos. A redação é deliberadamente narrativa — o objetivo é preservar como o problema realmente se manifestou, não só o dado técnico frio.  ## Incidente 2024-03-14 — Fila de recuperação de senha atrasada  Na manhã de uma q

**Por que isso gera uma resposta correta:** o chunk #1 contém, no próprio texto, a definição de Tier 0 ("qualquer indisponibilidade completa aqui é, por definição, SEV-1"), o nome do sistema (Auth Gateway) e o sintoma relatado quase literalmente. Um modelo de geração não precisa inferir nada, só resumir o que está na frente dele.

## Exemplo 2 — Sintoma enganoso, resolvido cruzando dois documentos

Este é mais interessante: "cartão recusado" *parece* um problema da operadora de cartão, fora do controle da NimbusFlow, e foi exatamente esse o erro cometido na triagem real (ver postmortem). O retrieval precisa trazer contexto suficiente pra não repetir esse erro.

**Esperado:** o **Catálogo de Sistemas Críticos** (Payment Gateway, que descreve esse sintoma como "um dos mais enganosos do catálogo") e o **postmortem do incidente 2024-06-02**, que registra a causa raiz real e a lição aprendida.

In [18]:
show_chunks(retrieve("cartão sendo recusado sem motivo aparente"))

[3.33] 04-postmortems-incidentes-anteriores.md — Incidente 2024-06-02 — Cartões recusados sem motivo aparente
## Incidente 2024-06-02 — Cartões recusados sem motivo aparente  Durante uma tarde de sexta-feira, o volume de chamados de suporte sobre "meu cartão foi recusado" subiu cerca de 4 vezes acima do normal. A princípio, o time assumiu que era um problema pontual do lado das adquirentes de cartão, fora do controle da NimbusFlow, e classificou como SEV-2 apenas por volume de reclamação. A investigação revelou que era, na v

[2.82] 02-catalogo-sistemas-criticos.md — Tier 0 — Caminho crítico de compra
## Tier 0 — Caminho crítico de compra  Sistemas cuja falha total impede diretamente uma venda de acontecer. Qualquer indisponibilidade completa aqui é, por definição, SEV-1.  **Auth Gateway** — responsável por login, criação de conta, recuperação de senha e emissão de tokens de sessão. Fica na frente de praticamente todo o resto: se ele cai, o cliente não consegue nem ver o próprio carrin

**Por que isso gera uma resposta correta:** os dois primeiros chunks se reforçam: o catálogo explica *por que* esse sintoma engana, e o postmortem mostra que já aconteceu antes, com a causa raiz real (timeout no Payment Gateway, não a operadora) e a lição registrada. Uma geração apoiada nesses dois chunks responde não só "qual sistema", mas "não confie na primeira impressão", o tipo de resposta que só é possível porque *dois* documentos concordam.

## Exemplo 3 — Baixa severidade esperada (retrieval não deve superestimar)

Tão importante quanto identificar um SEV-1 é o retrieval **não** empurrar o modelo pra uma resposta alarmista quando o caso é banal. Frase deliberadamente simples, sem termo técnico compartilhado com o manual.

**Esperado:** cair na seção **Tier 2 — Suporte operacional e interno**, em que o Notification Service é descrito como raramente urgente.

In [19]:
show_chunks(retrieve("fila de emails de boas-vindas atrasada uns 5 minutos"))

[2.22] 02-catalogo-sistemas-criticos.md — Tier 2 — Suporte operacional e interno
## Tier 2 — Suporte operacional e interno  Sistemas cuja falha não é percebida pelo cliente final em tempo real, mas afeta operação, comunicação ou times internos.  **Notification Service** — dispara emails e notificações push: confirmação de pedido, atualização de status de entrega, campanhas de marketing. Uma fila atrasada aqui é desconfortável mas raramente urgente — a única exceção relevante é quando o atraso afe

[1.97] 04-postmortems-incidentes-anteriores.md — Postmortems de Incidentes Anteriores — NimbusFlow
# Postmortems de Incidentes Anteriores — NimbusFlow  **Propósito:** registro histórico de incidentes já classificados e resolvidos, usado como referência para triagem de casos parecidos. A redação é deliberadamente narrativa — o objetivo é preservar como o problema realmente se manifestou, não só o dado técnico frio.  ## Incidente 2024-03-14 — Fila de recuperação de senha atrasada  Na manhã de u

**Por que isso gera uma resposta correta:** o chunk #1 já contrasta o caso comum ("fila atrasada... raramente urgente") com a única exceção real (email de recuperação de senha), então a geração consegue tanto classificar como baixa severidade quanto explicar, se perguntado, quando esse mesmo sistema *deixaria* de ser trivial.

## Exemplo 4 — Regra de exceção, não a regra geral

Este exemplo testa se o retrieval encontra a exceção certa, e não só a regra geral mais óbvia. Fora de época promocional, checkout lento com pouco volume tenderia a SEV-2. Mas em campanha (Black Friday), a lógica se inverte.

**Esperado:** a seção **"Exceções ligadas a horário e sazonalidade"** do runbook, que descreve as duas exceções (noturna e promocional) lado a lado.

In [20]:
show_chunks(retrieve("checkout lento durante a Black Friday, é pra tratar como grave mesmo sendo pouco volume?"))

[2.80] 01-runbook-triagem-incidentes.md — Exceções ligadas a horário e sazonalidade
## Exceções ligadas a horário e sazonalidade  Fora do horário comercial (antes das 8h ou depois das 20h, horário de Brasília), a barra para abrir SEV-1 sobe: um problema que durante o dia seria classificado como SEV-1 por afetar checkout parcialmente pode, à noite, ser tratado como SEV-2 se o volume de tráfego real no momento for baixo o suficiente para que o impacto absoluto (não percentual) seja pequeno. A pessoa d

[2.36] 05-guia-decisao-casos-limite.md — "Não sei se isso é o sistema X ou o sistema Y"
## "Não sei se isso é o sistema X ou o sistema Y"  É comum um sintoma reportado pelo cliente não indicar claramente qual sistema técnico está envolvido. Um bom hábito é seguir o sintoma até o efeito mais próximo do cliente, não até a causa técnica mais provável — a classificação de severidade depende do que o cliente está impedido de fazer, não de qual serviço internamente está com bug. Por exemplo, "nã

**Por que isso gera uma resposta correta:** o chunk #1 sozinho já contém as duas exceções (noturna reduz severidade, promocional aumenta) no mesmo parágrafo, então a geração não corre o risco de aplicar só a regra geral (5%-30% = SEV-2) e ignorar que estamos em campanha. A folga para o 2º colocado (2.80 vs 2.36) mostra que o retrieval está confiante de que achou a exceção certa, não só "algo sobre checkout".

## Exemplo 5 — Puxando um precedente histórico específico

Boa parte do valor de um RAG de suporte é não fazer o time repetir o mesmo raciocínio do zero a cada incidente parecido. Aqui a pergunta não descreve um sintoma técnico, ela pede diretamente por um precedente.

**Esperado:** o **postmortem do incidente 2024-03-14** (fila de recuperação de senha atrasada por campanha de marketing), que é praticamente o mesmo cenário perguntado.

In [21]:
show_chunks(retrieve("cliente reclamando que não recebe o email de recuperação de senha"))

[2.71] 04-postmortems-incidentes-anteriores.md — Postmortems de Incidentes Anteriores — NimbusFlow
# Postmortems de Incidentes Anteriores — NimbusFlow  **Propósito:** registro histórico de incidentes já classificados e resolvidos, usado como referência para triagem de casos parecidos. A redação é deliberadamente narrativa — o objetivo é preservar como o problema realmente se manifestou, não só o dado técnico frio.  ## Incidente 2024-03-14 — Fila de recuperação de senha atrasada  Na manhã de uma quinta-feira comum,

[2.65] 02-catalogo-sistemas-criticos.md — Tier 0 — Caminho crítico de compra
## Tier 0 — Caminho crítico de compra  Sistemas cuja falha total impede diretamente uma venda de acontecer. Qualquer indisponibilidade completa aqui é, por definição, SEV-1.  **Auth Gateway** — responsável por login, criação de conta, recuperação de senha e emissão de tokens de sessão. Fica na frente de praticamente todo o resto: se ele cai, o cliente não consegue nem ver o próprio carrinho salvo, m

**Por que isso gera uma resposta correta:** repare que o `section_title` do chunk #1 mostra só o cabeçalho do documento ("Postmortems de Incidentes Anteriores — NimbusFlow") isso é só um detalhe de metadado do chunk, mas o **conteúdo completo do chunk** (não o preview) inclui o incidente 2024-03-14 inteiro: mesma causa raiz (fila do Notification Service represada por campanha), mesma reclassificação de SEV-3 para SEV-2, e a resolução aplicada. A geração tem tudo que precisa pra responder "sim, já aconteceu, foi assim que resolvemos" mesmo com o rótulo do chunk sendo enganoso.

## Exemplo 6 — Fora da triagem: pergunta de processo

Vale mostrar que o retrieval não serve só pra classificar severidade, ele também responde perguntas de processo/protocolo, que vivem num documento totalmente diferente (Manual de Escalonamento, não o Runbook ou o Catálogo).

**Esperado:** a seção **"Quando liderança é convocada ativamente"**, que lista os três critérios exatos para chamar liderança pra dentro do war room.

In [22]:
show_chunks(retrieve("em que situações a liderança precisa entrar na guerra room"))

[3.14] 03-manual-escalonamento-plantao.md — Quando liderança é convocada ativamente
## Quando liderança é convocada ativamente  Liderança (Diretoria de Engenharia e, dependendo do caso, Jurídico ou Comunicação) é chamada ativamente para dentro do war room, e não apenas notificada, em três situações específicas: incidente classificado como SEV-1 que já dura mais de 45 minutos sem sinal de resolução iminente; qualquer incidente envolvendo risco de vazamento de dados de cliente, independentemente da du

[2.51] 03-manual-escalonamento-plantao.md — Protocolo de war room
## Protocolo de war room  Ao abrir um war room, três papéis são preenchidos, nessa ordem de prioridade:  1. **Incident Commander (IC):** coordena a resposta, decide se e quando escalar mais gente, e é a única pessoa autorizada a declarar o incidente encerrado. O IC não precisa (e idealmente não deve) ser a pessoa mexendo no código durante o incidente — a função é de coordenação, não de execução técnica. 2. **Comms Le

[2.16]

**Por que isso gera uma resposta correta:** o chunk #1 traz os três critérios completos e enumerados (SEV-1 acima de 45 minutos, risco de vazamento de dados, e mais de 50 menções públicas em uma hora) em um único parágrafo autocontido, com a maior folga de todos os exemplos (3.14 vs 2.51). É o tipo de resposta que uma geração consegue citar quase literalmente, sem parafrasear demais e sem risco de esquecer um dos três critérios.

## Conclusão

Nos 6 exemplos acima, o chunk que contém a resposta correta ficou em **1º lugar no `top_k=5`** e na maioria dos casos com folga clara para o 2º colocado, o que indica confiança do retrieval (e não um empate arbitrário decidido pelo reranker). Isso cobre um leque proposital de capacidades:

| # | Capacidade testada | Documento(s) |
|---|---|---|
| 1 | Sintoma óbvio → sistema/severidade óbvios | Catálogo |
| 2 | Cruzar sintoma enganoso com histórico real | Catálogo + Postmortems |
| 3 | Não superestimar severidade baixa | Catálogo |
| 4 | Achar a exceção certa, não só a regra geral | Runbook |
| 5 | Recuperar precedente histórico específico | Postmortems |
| 6 | Responder pergunta de processo, não só triagem | Manual de Escalonamento |

Com isso validado, o próximo passo natural é plugar um LLM de geração em cima desses mesmos `top_k` chunks. Agora o retrieval já não é mais o gargalo.